In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle
import tensorflow as tf

data = pd.read_csv('Churn_Modelling.csv', encoding='latin-1')
data.head()


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [2]:
#Data Preprocessing
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)

#Encode Gender Column
label_encode_gender = LabelEncoder()
data['Gender'] = label_encode_gender.fit_transform(data['Gender'])


In [3]:
#OneHotEncode Geography Column
from sklearn.preprocessing import OneHotEncoder
one_hot_encode_geography = OneHotEncoder()
geo_encoder = one_hot_encode_geography.fit_transform(data[['Geography']])
# one_hot_encode_geography.get_feature_names_out(['Geography'])
geography_df = pd.DataFrame(geo_encoder.toarray(), columns=one_hot_encode_geography.get_feature_names_out(['Geography']))

#Combine both dataframes
data = pd.concat([data.drop('Geography',axis = 1),geography_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [4]:
#Save the encoders and scaler
with open('label_encode_gender.pkl','wb') as file:
    pickle.dump(label_encode_gender,file)

with open('one_hot_encode_geography.pkl','wb') as file:
    pickle.dump(one_hot_encode_geography,file)


In [5]:
#Split the data into features and target variable
x = data.drop('Exited', axis=1) #Features
y = data['Exited'] #Target variable

#Split the data into train and test dataset
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

#Scale these features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

#Save the scalers
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [6]:
#ANN implementation
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import datetime

#Build the ANN model
model = Sequential(
    [
        Dense(64,activation='relu',input_shape=(x_train.shape[1],)), #Hidden layer-1
        Dense(32,activation='relu'), #Hiddel layer-2
        Dense(1,activation='sigmoid')
    ]
)

#model.summary()
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss = tf.keras.losses.BinaryCrossentropy()

#Compile the model
model.compile(optimizer=opt,loss=loss,metrics=['accuracy','Precision','Recall'])

g:\new-proj\myenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
#Setup tensorboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dirs = 'logs/fit/' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dirs, histogram_freq=1)


In [8]:
#Setup early stopping
early_stopping = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [9]:
#Train the model
history = model.fit(
    x_train, y_train, validation_data=(x_test,y_test),epochs=100,
    callbacks=[early_stopping,tensorboard_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - Precision: 0.6913 - Recall: 0.3583 - accuracy: 0.8353 - loss: 0.4014 - val_Precision: 0.7414 - val_Recall: 0.4377 - val_accuracy: 0.8595 - val_loss: 0.3490
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - Precision: 0.7336 - Recall: 0.4489 - accuracy: 0.8533 - loss: 0.3536 - val_Precision: 0.8025 - val_Recall: 0.3308 - val_accuracy: 0.8525 - val_loss: 0.3570
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.7429 - Recall: 0.4641 - accuracy: 0.8569 - loss: 0.3487 - val_Precision: 0.8278 - val_Recall: 0.3181 - val_accuracy: 0.8530 - val_loss: 0.3452
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.7652 - Recall: 0.4519 - accuracy: 0.8589 - loss: 0.3425 - val_Precision: 0.6937 - val_Recall: 0.5013 - val_accuracy: 0.8585 - val_loss: 0.3629
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.7675 - Recall: 0.4678 - accuracy: 0.8615 - loss: 0.3405 - val_Precision: 0.6156 - val_R

In [10]:
model.save('model.h5')

In [11]:
#Load Tensorboard Extension
%load_ext tensorboard
#%tensorboard --logdir logs/fit

In [12]:
#Load the pickle file
import pickle
from tensorflow.keras.models import load_model

model = load_model('model.h5')
with open('label_encode_gender.pkl','rb') as file:
    label_encode_gender = pickle.load(file)
    
with open('one_hot_encode_geography.pkl','rb') as file:
    one_hot_encode_geography = pickle.load(file)
    
with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [13]:
#Sample Input Data
sample_input = {
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [14]:
geo_encoder = one_hot_encode_geography.transform([[sample_input['Geography']]])
geography_df = pd.DataFrame(geo_encoder.toarray(), columns=one_hot_encode_geography.get_feature_names_out(['Geography']))
geography_df

g:\new-proj\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [15]:
#Combine one-hot encoded columns with input data
sample_input_df = pd.DataFrame([sample_input])

sample_input_df['Gender'] = label_encode_gender.transform(sample_input_df['Gender'])

sample_input_df = pd.concat([sample_input_df.reset_index(drop=True),geography_df],axis=1)
sample_input_df.drop('Geography', axis=1, inplace=True)
#sample_input = pd.concat([sample_input.drop('Geography',axis=1),geography_df],axis=1)
sample_input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [16]:
#Scaling the input data
input_scaled = scaler.transform(sample_input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [114]:
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


array([[0.0281316]], dtype=float32)